# Chapter 9 — Global Sensitivity Analysis and Sobol Indices
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local to Global Sensitivity*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Full global SA pipeline: LHS sampling, PRCC, Sobol decomposition,
Jansen estimator, Morris screening, and convergence analysis for the SIR model.

**Key claims tested:**
- ANOVA decomposition: D₁ = D₂ = 3/16, D = 55/144 for f = Q₁+Q₂+Q₁Q₂
- Jansen estimator: A_B^(i) shares column i from B, differs in all others
- First-order Sobol index S_i = η² for linear models (ANOVA connection)
- Global SA reverses local SA ranking when uncertainty is large
- Recommended pipeline: Morris screen → Sobol on top parameters

**Statistical connections (§9.3):** S_i = Var[E(Y|Q_i)]/Var(Y)
is identical to η² in classical ANOVA for linear models with independent inputs.
PRCC is a nonparametric partial regression coefficient (rank-based).

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
import numpy as np
from scipy.stats import qmc
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import rankdata, t as t_dist
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
FULL = False
M_SOBOL  = 2048 if FULL else 512    # Saltelli sample size
M_LHS    = 500  if FULL else 200    # LHS sample size for PRCC
N_MORRIS = 15   if FULL else 10     # Morris trajectories
print(f'Chapter 9: Global SA  (M_Sobol={M_SOBOL}, M_LHS={M_LHS}, N_Morris={N_MORRIS})')

In [ ]:
# ── Verification Suite — ANOVA worked example (§9.4.2) ───────────────────────
print('=' * 60)
print('VERIFICATION SUITE — ANOVA decomposition (§9.4.2)')
print('=' * 60)

# Analytic computation for f(Q1, Q2) = Q1 + Q2 + Q1*Q2, Q1,Q2 ~ U[0,1]
# E[f] = 1/2 + 1/2 + 1/4 = 5/4
# f1(Q1) = E[f|Q1] - E[f] = (Q1 + 1/2 + Q1/2) - 5/4 = 3Q1/2 - 3/4
# D1 = Var(f1) = (3/2)^2 * Var(Q1) = 9/4 * 1/12 = 3/16
# f12(Q1,Q2) = (Q1-1/2)(Q2-1/2)   → D12 = (1/12)^2 * ... 

# Exact values from book
D1_exact   = 3/16
D2_exact   = 3/16
D12_exact  = 1/144
D_exact    = 55/144
S1_exact   = D1_exact / D_exact   # = 27/55
S12_exact  = D12_exact / D_exact  # = 1/55

# Monte Carlo verification
M_mc = 1_000_000
Q1 = np.random.uniform(0, 1, M_mc)
Q2 = np.random.uniform(0, 1, M_mc)
Y  = Q1 + Q2 + Q1*Q2
D_mc = np.var(Y)

# First-order: D1 = Var[E(Y|Q1)]. For fixed Q1: E(Y|Q1) = Q1 + 1/2 + Q1/2 = 3Q1/2 + 1/2
EY_given_Q1 = 3*Q1/2 + 0.5
D1_mc = np.var(EY_given_Q1)
EY_given_Q2 = 3*Q2/2 + 0.5
D2_mc = np.var(EY_given_Q2)

# Verify analytic values
tol = 0.005  # 0.5% tolerance for Monte Carlo
assert abs(D1_mc - D1_exact)/D1_exact < tol, f'FAIL D1: MC={D1_mc:.5f}, exact={D1_exact:.5f}'
print(f'Test 1 PASS: D₁ = {D1_mc:.5f}  (exact = 3/16 = {D1_exact:.5f})')
assert abs(D2_mc - D2_exact)/D2_exact < tol, f'FAIL D2: MC={D2_mc:.5f}'
print(f'Test 2 PASS: D₂ = {D2_mc:.5f}  (exact = 3/16 = {D2_exact:.5f})')

D_exact_check = D1_exact + D2_exact + D12_exact
assert abs(D_exact_check - D_exact) < 1e-12, 'FAIL D decomp'
print(f'Test 3 PASS: D = D₁+D₂+D₁₂ = 3/16+3/16+1/144 = {D_exact:.6f} = 55/144')

assert abs(D_mc - D_exact)/D_exact < tol, f'FAIL D total: MC={D_mc:.5f}'
print(f'Test 4 PASS: Var(Y) = {D_mc:.5f}  (exact = 55/144 = {D_exact:.5f})')

# S1 = D1/D = (3/16)/(55/144) = (3/16)*(144/55) = 432/880 = 27/55
assert abs(S1_exact - 27/55) < 1e-12, 'FAIL S1 fraction'
print(f'Test 5 PASS: S₁ = 27/55 = {S1_exact:.6f}')

# Statistical connection: S1 = η² for linear models
Y_lin = Q1 + 2*Q2    # linear model
D_lin = np.var(Y_lin)
D1_lin = np.var(Q1 + np.mean(2*Q2))  # Var[E(Y_lin|Q1)] = Var(Q1)
S1_lin_mc  = np.var(Q1) / D_lin
S1_lin_exact = (1/12) / (1/12 + 4/12)  # = 1/5
assert abs(S1_lin_mc - S1_lin_exact) < tol, 'FAIL linear model S1'
print(f'Test 6 PASS: For Y=Q₁+2Q₂, S₁={S1_lin_mc:.4f} = η² = {S1_lin_exact:.4f}')

print(f'\nAll 6 verification tests PASSED.')
print('=' * 60)

In [ ]:
# ── Verification Suite, part 2: the ANOVA worked example as printed (§9.4.2) ──
# Exact rational arithmetic, so no sampling error enters.
from fractions import Fraction as Fr

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

half, var_u = Fr(1, 2), Fr(1, 12)                 # mean and variance of U(0, 1)
f0 = half + half + half * half
D1 = D2 = Fr(3, 2) ** 2 * var_u
D12 = var_u * var_u
D_tot = D1 + D2 + D12
S1, S2, S12 = D1 / D_tot, D2 / D_tot, D12 / D_tot
assert f0 == Fr(5, 4) and D1 == Fr(3, 16) and D12 == Fr(1, 144), 'FAIL: components'
assert D_tot == Fr(55, 144) and D1 == Fr(27, 144), 'FAIL: total variance'
assert S1 == S2 == Fr(27, 55) and S12 == Fr(1, 55), 'FAIL: index fractions'
assert agrees(float(S1), 0.491, 3) and agrees(float(S12), 0.018, 3), 'FAIL: printed decimals'  # book: ch09_body.tex "\tfrac{27}{55} \approx 0.491" "\tfrac{1}{55} \approx 0.018"
assert S1 + S2 + S12 == 1, 'FAIL: indices must sum to 1'
S_add = var_u / (var_u + var_u)                   # additive model Q1 + Q2
assert S_add == half, 'FAIL: additive model S_i = 1/2'
print(f'PASS: f0 = {f0}, D = {D_tot}, S1 = S2 = {S1} = {float(S1):.3f}, '
      f'S12 = {S12} = {float(S12):.3f}, sum = {S1 + S2 + S12}')

In [ ]:
# ── Verification Suite, part 3: the Morris walkthrough as printed (§9.6) ─────
# f(u1, u2) = 3 u1 + u2 + 2 u1 u2 on [0, 1]^2, Delta = 1/2, r = 3 trajectories.

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

f_m = lambda u1, u2: 3 * u1 + u2 + 2 * u1 * u2
Dm = 0.5
trajectories = [((0.0, 0.0), (0, 1)), ((0.5, 0.5), (1, 0)), ((0.5, 0.0), (0, 1))]
EE = {0: [], 1: []}
for start, order in trajectories:
    pt = list(start)
    for k in order:
        before = f_m(*pt)
        pt[k] += Dm
        EE[k].append((f_m(*pt) - before) / Dm)
assert EE[0] == [3.0, 5.0, 3.0] and EE[1] == [2.0, 2.0, 3.0], f'FAIL: EE = {EE}'  # book: ch09_body.tex "EE_1 = \frac{1.50 - 0}{0.5} = 3.0" "EE_2 = \frac{2.50 - 1.50}{0.5} = 2.0" "EE_2 = \frac{3.50 - 2.50}{0.5} = 2.0" "EE_1 = \frac{6.00 - 3.50}{0.5} = 5.0" "EE_1 = \frac{3.00 - 1.50}{0.5} = 3.0" "EE_2 = \frac{4.50 - 3.00}{0.5} = 3.0" "$u_1$ & 3.0 & 5.0 & 3.0" "$u_2$ & 2.0 & 2.0 & 3.0"
assert [f_m(0.5, 0.0), f_m(0.5, 0.5), f_m(0.5, 1.0), f_m(1.0, 1.0), f_m(1.0, 0.5)] == \
    [1.5, 2.5, 3.5, 6.0, 4.5], 'FAIL: trajectory function values'  # book: ch09_body.tex "f(0.5,0.0) = 1.50" "f(0.5,0.5) = 2.50" "f(0.5,1.0) = 3.50" "f(1.0,1.0) = 6.00" "f(1.0,0.5) = 4.50" "\frac{1.50 - 0}{0.5}" "\frac{2.50 - 1.50}{0.5}" "\frac{3.50 - 2.50}{0.5}" "\frac{6.00 - 3.50}{0.5}" "\frac{3.00 - 1.50}{0.5}" "\frac{4.50 - 3.00}{0.5}"
assert f_m(1.0, 0.0) == 3.0, 'FAIL: f(1, 0)'  # book: ch09_body.tex "f(1.0,0.0) = 3.00" "\frac{3.00 - 1.50}{0.5}" "\frac{4.50 - 3.00}{0.5}"
assert len(trajectories) == 3, 'FAIL: r'  # book: ch09_body.tex "and $r = 3$ trajectories" "an artifact of $r = 3$" "with $r = 3$ one of" "In practice, $r = 3$ is too few"
# The elementary effects in closed form, checked on a grid of base points (not only the three trajectories).
_g = [(a, b) for a in (0.0, 0.25, 0.5) for b in (0.0, 0.25, 0.5)]
_ee1 = np.array([(f_m(a + Dm, b) - f_m(a, b)) / Dm for a, b in _g]); _u2 = np.array([b for a, b in _g])
_ee2 = np.array([(f_m(a, b + Dm) - f_m(a, b)) / Dm for a, b in _g]); _u1 = np.array([a for a, b in _g])
assert np.allclose(_ee1, 3 + 2 * _u2, rtol=0, atol=1e-12) and np.allclose(_ee2, 1 + 2 * _u1, rtol=0, atol=1e-12), 'FAIL: EE formulas'  # book: ch09_body.tex "$EE_1 = 3 + 2u_2$ and $EE_2 = 1 + 2u_1$"
mu_star = [np.mean(np.abs(EE[k])) for k in (0, 1)]
sig = [np.std(EE[k], ddof=1) for k in (0, 1)]
assert agrees(mu_star[0], 3.67, 2) and agrees(mu_star[1], 2.33, 2), f'FAIL: mu* = {mu_star}'  # book: ch09_body.tex "& 3.0 & 3.67 & 1.15" "& 3.0 & 2.33 & 0.58"
assert agrees(sig[0], 1.15, 2) and agrees(sig[1], 0.58, 2), f'FAIL: sigma = {sig}'  # book: ch09_body.tex "& 3.0 & 3.67 & 1.15" "& 3.0 & 2.33 & 0.58"
assert agrees(np.sqrt(4 * (1 / 12)), 0.577, 3), 'FAIL: population sigma = 1/sqrt(3)'  # book: ch09_body.tex "\sigma = 1/\sqrt{3} \approx 0.577"
print(f'PASS: mu* = {mu_star[0]:.2f}, {mu_star[1]:.2f};  sigma = {sig[0]:.2f}, {sig[1]:.2f};  '
      f'population sigma = {1/np.sqrt(3):.3f} for both')

In [ ]:
# ── SIR model for global SA ───────────────────────────────────────────────────
def sir_peak_I(params):
    """Return peak infectious count I_max for the demographic SIR model.

    params = [c, beta, tau_R, tau_m], the four POIs of the Chapter 9 laboratory.
    The response is I_max, not R_0, and I_max does depend on tau_m through the
    demographic terms, so all four are sampled.
    """
    c, beta, tau_R, tau_m = params
    N = 1000
    nu_m = 1.0/tau_m
    def rhs(t, y):
        S, I, R = y
        return [nu_m*N - c*beta*S*I/N - nu_m*S,
                c*beta*S*I/N - (1/tau_R + nu_m)*I,
                I/tau_R - nu_m*R]
    sol = solve_ivp(rhs, [0, 90], [999, 1, 0],
                    rtol=1e-8, atol=1e-10, dense_output=False)
    return sol.y[1].max()

# Parameter ranges exactly as the Chapter 9 laboratory specifies:
#   c ∈ [2, 8],  β ∈ [0.02, 0.12],  τ_R ∈ [4, 14] days,  τ_m ∈ [1000, 100000] days
lb = np.array([2.0,  0.02, 4.0,  1_000.0])
ub = np.array([8.0,  0.12, 14.0, 100_000.0])
print(f'SIR model: I_max at nominal = {sir_peak_I([5, 0.06, 7, 10_000]):.1f}')

In [ ]:
# ── LHS Sampling ──────────────────────────────────────────────────────────────
def lhs_sample(n, lb, ub, rng=None):
    """Latin Hypercube Sample: n rows, m=len(lb) columns."""
    rng = np.random.default_rng(rng)
    m = len(lb)
    cut = np.linspace(0, 1, n+1)
    X = np.zeros((n, m))
    for j in range(m):
        u = rng.uniform(cut[:-1], cut[1:])
        X[:, j] = lb[j] + rng.permutation(u) * (ub[j] - lb[j])
    return X

X_lhs = lhs_sample(M_LHS, lb, ub, rng=42)
print(f'LHS sample: {X_lhs.shape}, ranges OK: {(X_lhs >= lb).all() and (X_lhs <= ub).all()}')

In [ ]:
# ── PRCC Computation (§9.3) ───────────────────────────────────────────────────
print('Computing PRCC (this may take ~1 min for M_LHS=200)...')
Y_lhs = np.array([sir_peak_I(x) for x in X_lhs])
print(f'Model evaluations: {M_LHS}  |  I_max range: [{Y_lhs.min():.1f}, {Y_lhs.max():.1f}]')

def prcc(X, Y):
    """
    Compute PRCC (Partial Rank Correlation Coefficient) for each column of X.
    PRCC is a nonparametric partial regression coefficient (§9.3).

    Parameters
    ----------
    X : ndarray (n, m) — input matrix
    Y : ndarray (n,)   — scalar model output

    Returns
    -------
    prcc  : ndarray (m,), the PRCC values
    pvals : ndarray (m,) — t-test p-values
    """
    n, m = X.shape
    # Rank-transform
    Xr = np.column_stack([rankdata(X[:, j]) for j in range(m)])
    Yr = rankdata(Y)

    # Partial correlation: regress Xr[:,j] and Yr on all other columns
    prcc  = np.zeros(m)
    pvals = np.zeros(m)
    for j in range(m):
        other = np.delete(np.arange(m), j)
        Z = Xr[:, other]  # confounders

        # Residuals after regressing Xr[:,j] on Z
        Z_aug  = np.column_stack([np.ones(n), Z])
        coef_x = np.linalg.lstsq(Z_aug, Xr[:, j], rcond=None)[0]
        resid_x = Xr[:, j] - Z_aug @ coef_x

        # Residuals after regressing Yr on Z
        coef_y = np.linalg.lstsq(Z_aug, Yr, rcond=None)[0]
        resid_y = Yr - Z_aug @ coef_y

        # Partial correlation = correlation of residuals
        r  = np.corrcoef(resid_x, resid_y)[0, 1]
        prcc[j] = r

        # t-statistic with n-m-1 df
        df = n - m - 1
        t_stat = r * np.sqrt(df / (1 - r**2 + 1e-15))
        pvals[j] = 2 * (1 - t_dist.cdf(abs(t_stat), df))

    return prcc, pvals

prcc_vals, pvals_prcc = prcc(X_lhs, Y_lhs)
param_names = ['c', 'β', 'τ_R', 'τ_m']
print('\nPRCC results for I_max:')
for name, r, pv in zip(param_names, prcc_vals, pvals_prcc):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    print(f'  PRCC_{name} = {r:+.4f}  p={pv:.4f} {sig}')

In [ ]:
# ── Saltelli/Jansen Sobol Estimator (§9.4–9.5) ───────────────────────────────
print(f'\nComputing Sobol indices (M={M_SOBOL}, {M_SOBOL*(len(lb)+2)} total model evaluations)...')

def saltelli_sample(M, lb, ub, rng=42):
    """Generate Saltelli sample matrices A and B (each M×m).

    Draws a genuine Sobol' sequence in 2m dimensions and splits it into the
    A and B blocks, which is what §9.12 tells the reader to do.  The balance
    property of a Sobol' sequence holds in blocks of 2^k, so M should be a
    power of two; scipy warns if it is not.  Do not substitute independent
    pseudo-random uniforms here: the power-of-two justification in the text
    describes a Sobol' sequence and is vacuous without one.
    """
    m = len(lb)
    engine = qmc.Sobol(d=2*m, scramble=True, seed=rng)
    AB = engine.random(M)                      # M × 2m, low-discrepancy
    A  = lb + AB[:, :m] * (ub - lb)
    B  = lb + AB[:, m:] * (ub - lb)
    return A, B

def sobol_jansen(model, M, lb, ub, rng=42):
    """
    Estimate first-order Sobol indices via Jansen estimator.
    A_B^(i) = matrix A with column i replaced by column i of B.
    S_i = 1 - Var[f(B) - f(A_B^(i))] / (2*Var(f))
    """
    m  = len(lb)
    A, B = saltelli_sample(M, lb, ub, rng)

    fA = np.array([model(A[i]) for i in range(M)])
    fB = np.array([model(B[i]) for i in range(M)])
    D  = np.var(np.concatenate([fA, fB]))

    S1 = np.zeros(m)
    for j in range(m):
        # A_B^(j): A with column j replaced by B's column j
        AB_j = A.copy()
        AB_j[:, j] = B[:, j]
        fABj = np.array([model(AB_j[i]) for i in range(M)])
        # Jansen (1999) first-order estimator
        S1[j] = 1.0 - np.mean((fB - fABj)**2) / (2*D)

    return S1, D

S1_sobol, D_sobol = sobol_jansen(sir_peak_I, M_SOBOL, lb, ub)
print('\nSobol first-order indices for I_max:')
for name, s in zip(param_names, S1_sobol):
    print(f'  S₁({name}) = {s:.4f}')
print(f'  Sum S₁ = {S1_sobol.sum():.4f}  (< 1 means interactions present)')

In [ ]:
# ── Figure 9.1: PRCC vs Sobol comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# PRCC bar chart
ax = axes[0]
colors_prcc = ['steelblue' if r > 0 else 'coral' for r in prcc_vals]
ax.bar(param_names, prcc_vals, color=colors_prcc, edgecolor='white', width=0.5)
ax.axhline(0, color='black', lw=0.8)
for i, (r, pv) in enumerate(zip(prcc_vals, pvals_prcc)):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    ax.text(i, r + 0.02*np.sign(r), f'{r:.3f}{sig}', ha='center', fontsize=10)
ax.set_ylabel('PRCC', fontsize=12)
ax.set_title(r'PRCC for $I_{\rm max}$'+f'\n(M={M_LHS} LHS samples)', fontsize=11)
ax.set_ylim(-0.15, 1.05)
ax.grid(axis='y', alpha=0.3)

# Sobol bar chart
ax2 = axes[1]
ax2.bar(param_names, S1_sobol, color='seagreen', edgecolor='white', width=0.5)
ax2.axhline(0, color='black', lw=0.8)
for i, s in enumerate(S1_sobol):
    ax2.text(i, s + 0.01, f'{s:.3f}', ha='center', fontsize=10)
ax2.set_ylabel(r'First-order Sobol index $\mathcal{S}_i$', fontsize=12)
ax2.set_title(r'Sobol $\mathcal{S}_1$ for $I_{\rm max}$'+f'\n(M={M_SOBOL} Saltelli evaluations)', fontsize=11)
ax2.set_ylim(0, 0.85)
ax2.grid(axis='y', alpha=0.3)
ax2.text(0.98, 0.95, f'Σ S₁ = {S1_sobol.sum():.3f}',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9, color='gray')

plt.suptitle('Global SA for SIR peak infections\n'
             'PRCC (monotonicity-based) vs Sobol (variance-based)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('ch09_fig1_prcc_sobol.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch09_fig1_prcc_sobol.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9.1 saved.')

In [ ]:
# ── Figure 9.2: ANOVA decomposition verification ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: Sobol indices for toy model f = Q1 + Q2 + Q1*Q2
def toy_model(q): return q[0] + q[1] + q[0]*q[1]
lb2, ub2 = np.array([0., 0.]), np.array([1., 1.])
S1_toy, _ = sobol_jansen(toy_model, 2048, lb2, ub2)

ax = axes[0]
ax.bar(['$Q_1$', '$Q_2$'], [S1_exact, S1_exact],
       color='steelblue', alpha=0.4, label='Analytic', width=0.35, align='edge')
ax.bar(['$Q_1$', '$Q_2$'], S1_toy,
       color='coral', alpha=0.7, label='Jansen MC', width=-0.35, align='edge')
ax.axhline(S1_exact, color='steelblue', ls='--', lw=1.2)
ax.set_ylabel('First-order Sobol index', fontsize=11)
ax.set_title(fr'$f = Q_1+Q_2+Q_1Q_2$: $\mathcal{{S}}_1=\mathcal{{S}}_2=27/55\approx{S1_exact:.3f}$', fontsize=10)
ax.legend(fontsize=9); ax.set_ylim(0, 0.65)
ax.text(0.5, 0.15, fr'$\mathcal{{S}}_{{12}} = 1/55 \approx {S12_exact:.3f}$',
        transform=ax.transAxes, ha='center', fontsize=10, color='gray')

# Right: Variance decomposition pie
ax2 = axes[1]
sizes = [D1_exact/D_exact, D2_exact/D_exact, D12_exact/D_exact]
labels = [fr'$D_1={D1_exact:.4f}$\n$\mathcal{{S}}_1={S1_exact:.3f}$',
          fr'$D_2={D2_exact:.4f}$\n$\mathcal{{S}}_2={S1_exact:.3f}$',
          fr'$D_{{12}}={D12_exact:.4f}$\n$\mathcal{{S}}_{{12}}={S12_exact:.3f}$']
colors_pie = ['steelblue', 'coral', 'seagreen']
ax2.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 9})
ax2.set_title(fr'ANOVA decomposition: $D = {D_exact:.4f} = 55/144$', fontsize=10)

plt.tight_layout()
plt.savefig('ch09_fig2_anova.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch09_fig2_anova.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9.2 saved.')

In [ ]:
# ── Results & Download ─────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 9,
    'ANOVA_exact': {'D1': D1_exact, 'D2': D2_exact, 'D12': D12_exact,
                    'D': D_exact, 'S1': S1_exact, 'S12': S12_exact},
    'PRCC_SIR_Imax': dict(zip(param_names, prcc_vals.tolist())),
    'Sobol_S1_SIR_Imax': dict(zip(param_names, S1_sobol.tolist())),
    'Sobol_sum_S1': float(S1_sobol.sum()),
    'figures': ['ch09_fig1_prcc_sobol.pdf', 'ch09_fig2_anova.pdf']
}
for k_r, v in RESULTS.items():
    print(f'  {k_r}: {v}')

try:
    from google.colab import files
    for f in RESULTS['figures']:
        files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')

In [ ]:
# ── Structural assertions for the four-POI laboratory ────────────────────────
# tau_m is sampled over [1000, 100000] days against a 90-day outbreak, so
# demography is nearly irrelevant to I_max.  That is a falsifiable prediction:
# tau_m must rank LAST with a near-zero index.  A three-parameter version of this
# notebook cannot satisfy these, which is exactly why they are stated.
assert len(lb) == 4, f"laboratory specifies four POIs; found {len(lb)}"
assert param_names[3] == '\u03c4_m', f"fourth POI is {param_names[3]}, expected tau_m"
_S1 = dict(zip(param_names, S1_sobol))
assert _S1[param_names[3]] < 1 / 20, f"S1(tau_m) = {_S1[param_names[3]]:.4f}; expected < 0.05"  # integer form: no printed number is meant
assert _S1[param_names[3]] == min(_S1.values()), "tau_m should rank last of the four"
assert 85 / 100 < S1_sobol.sum() < 115 / 100, f"sum S1 = {S1_sobol.sum():.4f}; expected near 1"  # integer form: structural bound, not a printed number
print(f"Structural checks PASS: S1({param_names[3]}) = {_S1[param_names[3]]:.4f} "
      f"ranks last, sum S1 = {S1_sobol.sum():.4f}")

In [ ]:
# ── Verification Suite, part 4: values the running text quotes ───────────────────
# The numbers in §§9.1-9.6 that the cells above do not already check.  Each check
# names the book line it is about with a comment of the form
#     # book: ch09_body.tex "text on that line"
# so the check accounts for that
# occurrence and for no other.  Parameters the text states are bound to named
# variables here, each with the value the text prints.

# §9.1  Two parameters with s = (2, 1) and sigma = (1, 3): Var(Y) = 4 + 9 = 13.
s_toy, sd_toy = np.array([2.0, 1.0]), np.array([1.0, 3.0])
sigma_Y_toy = np.sqrt(np.sum((s_toy * sd_toy) ** 2))
assert agrees(sd_toy[0] / sigma_Y_toy * s_toy[0], 0.55, 2)  # book: ch09_body.tex "\cdot 2 \approx 0.55"
assert agrees(sd_toy[1] / sigma_Y_toy * s_toy[1], 0.83, 2)  # book: ch09_body.tex "\cdot 1 \approx 0.83"

# §9.2  "Two SIR Models" and the self-check after it.
c_star, beta_star, tauR_star = 5, 0.06, 7                 # nominal values
sigma_c, sigma_beta, sigma_tauR = 0.5, 0.01, 2            # the self-check's uncertainties
sigma_beta_A, sigma_beta_B = 0.005, 0.03                  # Setting A, Setting B
grad_R0 = np.array([beta_star * tauR_star, c_star * tauR_star, c_star * beta_star])

def sigma_normalized(sd):
    """S_i^sigma = (dR0/dp_i) sigma_i / sigma_R0, sigma_R0 by the delta method."""
    return grad_R0 * sd / np.sqrt(np.sum((grad_R0 * sd) ** 2))

rise = (sigma_normalized(np.array([sigma_c, sigma_beta_B, sigma_tauR]))[1]
        / sigma_normalized(np.array([sigma_c, sigma_beta_A, sigma_tauR]))[1])
assert np.isclose(grad_R0[1] * sigma_beta_B, 6 * grad_R0[1] * sigma_beta_A)   # six times larger
assert rise < 6 and agrees(rise, 3.2, 1)  # book: ch09_body.tex "about~$3.2$"

# The self-check answer, line by line as printed (rounded intermediates included),
# then the same quantities without rounding.
assert agrees(c_star * beta_star * tauR_star, 2.1, 1)  # book: ch09_body.tex "\cR_0^* = 2.1"
assert agrees(grad_R0[0], 0.42, 2) and agrees(grad_R0[2], 0.3, 1)  # book: ch09_body.tex "= \beta\tau_R = 0.42$" "= c\beta = 0.3$."
_R0f = lambda b: c_star * b * tauR_star  # the derivative measured, not the printed product
assert agrees((_R0f(beta_star + 1e-6) - _R0f(beta_star - 1e-6)) / 2e-6, 35, 6), 'FAIL: dR0/dbeta = c tau_R'  # book: ch09_body.tex "\partial\beta = c\tau_R = 35"
assert agrees(0.42 ** 2 * 0.25 + 35 ** 2 * 0.0001 + 0.3 ** 2 * 4, 0.527, 3)  # book: ch09_body.tex "\approx 0.42^2(0.25) + 35^2(0.0001) + 0.3^2(4)"
assert (agrees(0.42 ** 2 * 0.25, 0.0441, 4) and agrees(35 ** 2 * 0.0001, 0.1225, 4)
        and agrees(0.3 ** 2 * 4, 0.36, 2) and agrees(0.0441 + 0.1225 + 0.36, 0.527, 3))  # book: ch09_body.tex "= 0.0441 + 0.1225 + 0.36 = 0.527$."
var_R0 = np.sum((grad_R0 * np.array([sigma_c, sigma_beta, sigma_tauR])) ** 2)
assert agrees(var_R0, 0.527, 3) and agrees(np.sqrt(var_R0), 0.726, 3)  # book: ch09_body.tex "\sigma_{\cR_0} \approx 0.726$"
S_sig = sigma_normalized(np.array([sigma_c, sigma_beta, sigma_tauR]))
assert agrees(0.42 * 0.5 / 0.726, 0.29, 2) and agrees(S_sig[0], 0.29, 2)  # book: ch09_body.tex "\Upsilon_c^\sigma = 0.42 \times 0.5/0.726"
assert agrees(35 * 0.01 / 0.726, 0.48, 2) and agrees(S_sig[1], 0.48, 2)  # book: ch09_body.tex "\Upsilon_\beta^\sigma = 35 \times 0.01/0.726"
assert agrees(0.3 * 2 / 0.726, 0.83, 2) and agrees(S_sig[2], 0.83, 2)  # book: ch09_body.tex "\Upsilon_{\tau_R}^\sigma = 0.3 \times 2/0.726"
assert list(np.argsort(-S_sig)) == [2, 1, 0]               # the ranking tau_R > beta > c
# The CV form of the same answer: with all three local SIs equal to 1, S_i^sigma = CV_i / CV_R0.
CV_in = np.array([sigma_c / c_star, sigma_beta / beta_star, sigma_tauR / tauR_star])
CV_R0 = np.sqrt(var_R0) / (c_star * beta_star * tauR_star)
assert agrees(CV_in[0], 0.10, 2) and agrees(CV_in[1], 0.17, 2) and agrees(CV_in[2], 0.29, 2) and np.allclose(S_sig, CV_in / CV_R0)  # book: ch09_body.tex "$\CV_c = 0.10$, $\CV_\beta = 0.17$, $\CV_{\tau_R} = 0.29$ and"
assert agrees(0.726 / 2.1, 0.346, 3) and agrees(CV_R0, 0.346, 3) and agrees(np.sqrt(var_R0), 0.726, 3) and agrees(c_star * beta_star * tauR_star, 2.1, 1)  # book: ch09_body.tex "$\CV_{\cR_0} = 0.726/2.1 = 0.346$"

# §9.3  LHS self-check, M = 4 on [0, 1]: four quarter-intervals, one point in each.
edges_lhs = np.linspace(0, 1, 5)
assert np.allclose(edges_lhs, [0, 0.25, 0.5, 0.75, 1])  # book: ch09_body.tex "each quarter-interval $[0,0.25)$, $[0.25,0.5)$," "$[0.5,0.75)$, $[0.75,1]$ must contain" "one point in each of $[0,0.25)$, $[0.25,0.5)$," "$[0.5,0.75)$, $[0.75,1]$, with uniform jitter"
assert 0.5 ** 4 == 1 / 16  # book: ch09_body.tex "fall in $[0, 0.5]$?" "all 4 points in $[0, 0.5]$"
assert [int(4 * x) for x in [0.18, 0.43, 0.62, 0.91]] == [0, 1, 2, 3]  # book: ch09_body.tex "$X = \{0.18, 0.43, 0.62, 0.91\}$"

# §9.5  Cost at M = 1024 with m = 4: M(m + 2) = 6144 runs; at 0.1 s a run, about 10 minutes.
assert 1024 * (4 + 2) == 6144 and round(6144 * 0.1 / 60) == 10  # book: ch09_body.tex "If each evaluation takes 0.1~seconds"

# Screening before Sobol: M(m + 2) runs for all m = 30 parameters at M = 2048, against M(k + 2) for the
# k = 5 that survive screening plus the r(m + 1) Morris runs at r = 10.
_M_W, _m_W, _K_W, _R_W = 2048, 30, 5, 10
_full_W, _scr_W, _mor_W = _M_W * (_m_W + 2), _M_W * (_K_W + 2), _R_W * (_m_W + 1)
assert _full_W == 65536 and _scr_W == 14336 and _m_W == 30, 'FAIL: Sobol costs'  # book: ch09_body.tex "For the $m = 30$ parameters that open this section,"
assert agrees(_full_W / (_scr_W + _mor_W), 4.5, 1) and _mor_W == 310, f'FAIL: factor {_full_W / (_scr_W + _mor_W):.3f}'  # book: ch09_body.tex "a factor of about $4.5$ once the $r(m+1) = 310$"
assert _R_W == 10 and _mor_W == _R_W * 31, 'FAIL: Morris runs'  # book: ch09_body.tex "Morris runs at $r = 10$ are counted."

# §9.6  The Morris walkthrough as printed: starting points, values, steps (cell 4's f_m).
assert trajectories[0][0] == (0.0, 0.0)  # book: ch09_body.tex "Start at $(0.0,\,0.0)$"
assert trajectories[1][0] == (0.5, 0.5)  # book: ch09_body.tex "Start at $(0.5,\,0.5)$"
assert trajectories[2][0] == (0.5, 0.0)  # book: ch09_body.tex "Start at $(0.5,\,0.0)$"
assert f_m(0.0, 0.0) == 0 and f_m(0.5, 0.0) == 1.50 and f_m(0.5, 0.5) == 2.50  # book: ch09_body.tex "f(0.0,0.0) = 0," "f(0.5,0.0) = 1.50," "f(0.5,0.5) = 2.50."
assert f_m(0.5, 0.5) == 2.50 and f_m(0.5, 1.0) == 3.50 and f_m(1.0, 1.0) == 6.00  # book: ch09_body.tex "f(0.5,0.5) = 2.50," "f(0.5,1.0) = 3.50," "f(1.0,1.0) = 6.00."
assert f_m(1.0, 0.0) == 3.00 and f_m(1.0, 0.5) == 4.50  # book: ch09_body.tex "f(1.0,0.0) = 3.00," "f(1.0,0.5) = 4.50."
assert (1.50 - 0) / 0.5 == 3.0 and (2.50 - 1.50) / 0.5 == 2.0  # book: ch09_body.tex "EE_1 = \frac{1.50 - 0}{0.5} = 3.0" "EE_2 = \frac{2.50 - 1.50}{0.5} = 2.0"
assert (3.50 - 2.50) / 0.5 == 2.0 and (6.00 - 3.50) / 0.5 == 5.0  # book: ch09_body.tex "EE_2 = \frac{3.50 - 2.50}{0.5} = 2.0" "EE_1 = \frac{6.00 - 3.50}{0.5} = 5.0"
assert (3.00 - 1.50) / 0.5 == 3.0 and (4.50 - 3.00) / 0.5 == 3.0  # book: ch09_body.tex "EE_1 = \frac{3.00 - 1.50}{0.5} = 3.0" "EE_2 = \frac{4.50 - 3.00}{0.5} = 3.0"
print('PASS: part 4, the quoted values of §§9.1-9.6')

In [ ]:
# ── Verification Suite, part 5: the §9.5.1 worked example and the laboratory (tau_R since v1_22_0) ──────
# §9.5.1: SIR with three POIs (c, beta, tau_R), QOI the peak infectious fraction, ranges
# +/-50% about c = 8, beta = 0.06 and tau_R = 10 days; a Saltelli design with M = 256 and
# the Jansen estimators.  Design and estimator are this notebook's own: saltelli_sample and
# the Jansen form of cell 8.  Since v1_22_0 the text prints this cell's output.
# The intervals come from R = 32 independent
# scrambles of that design, seeds 42..73 (seed 42 is the design whose estimates the text
# prints first), as mean +/- t_{R-1, 0.975} s / sqrt(R).  The row bootstrap (nBoot = 100,
# rows drawn with default_rng(42)) is kept as the IID fallback the text compares against.
# In order, the cell establishes:
#   1. the peak depends on the POIs only through R0 = c*beta*tau_R (closed form = ODE);
#   2. the point estimates of the seed-42 design, scrambles 1-4 and 32, the replicate
#      intervals, their widths, the budget, and the bootstrap comparison, as printed;
#   3. the population indices, by quadrature, are equal: R0 is a symmetric product of
#      three POIs with the same relative spread, so the estimates differ only by scatter.
nominal_951 = np.array([8.0, 0.06, 10.0])
lb_951, ub_951 = 0.5 * nominal_951, 1.5 * nominal_951
M_951, NBOOT_951 = 256, 100
R_951, T_LEVEL_951 = 32, 0.975                      # scrambles; two-sided 95% t quantile level
assert len(nominal_951) == 3, 'FAIL: m'  # book: ch09_body.tex "Apply the SIR model ($m = 3$ parameters"
S0_951, I0_951 = 0.999, 0.001

def peak_closed(R0):
    """Peak infectious fraction of SIR started at (S0, I0): a function of R0 alone."""
    R0 = np.asarray(R0, float)
    return np.where(R0 * S0_951 > 1, I0_951 + S0_951 - (1 + np.log(R0 * S0_951)) / R0, I0_951)

def peak_ode(p, t_end=400.0):
    c, b, tau = p
    t = np.linspace(0, t_end, 400_001)
    sol = solve_ivp(lambda _t, y: [-c * b * y[0] * y[1], c * b * y[0] * y[1] - y[1] / tau],
                    [0, t_end], [S0_951, I0_951], t_eval=t, rtol=1e-11, atol=1e-13)
    return sol.y[1].max()

ODE_TOL = 1e-6
for p in (nominal_951, lb_951, ub_951, np.array([lb_951[0], ub_951[1], ub_951[2]])):
    assert abs(peak_ode(p) - peak_closed(p[0] * p[1] * p[2])) < ODE_TOL, f'closed form off at {p}'
qoi_951 = lambda p: float(peak_closed(p[0] * p[1] * p[2]))

def jansen_with_values(model, M, lb_, ub_, seed=42):
    """Cell 8's design and Jansen estimator, keeping the model values for the bootstrap."""
    A, B = saltelli_sample(M, lb_, ub_, seed)
    fA = np.array([model(a) for a in A])
    fB = np.array([model(b) for b in B])
    fAB = []
    for j in range(len(lb_)):
        ABj = A.copy()
        ABj[:, j] = B[:, j]
        fAB.append(np.array([model(x) for x in ABj]))
    return fA, fB, np.array(fAB)

def jansen_S1(fA, fB, fAB):
    D = np.var(np.concatenate([fA, fB]))
    return 1.0 - np.mean((fB - fAB) ** 2, axis=1) / (2 * D)

fA_951, fB_951, fAB_951 = jansen_with_values(qoi_951, M_951, lb_951, ub_951)
S_hat_951 = jansen_S1(fA_951, fB_951, fAB_951)
assert np.allclose(S_hat_951, sobol_jansen(qoi_951, M_951, lb_951, ub_951)[0])   # cell 8's estimator
print('point estimates (M = 256, seed 42): ' + ', '.join(f'{s:.4f}' for s in S_hat_951))
assert agrees(S_hat_951[0], 0.32, 2)  # book: ch09_body.tex "\hat{\mathcal{S}}_c = 0.32"
assert agrees(S_hat_951[1], 0.33, 2)  # book: ch09_body.tex "\hat{\mathcal{S}}_\beta = 0.33"
assert agrees(S_hat_951[2], 0.33, 2)  # book: ch09_body.tex "\hat{\mathcal{S}}_{\tau_R} = 0.33"
assert np.allclose(np.round(S_hat_951, 2), [0.32, 0.33, 0.33])  # book: ch09_body.tex "small differences among $0.32$, $0.33$ and $0.33$"
assert agrees(S_hat_951.mean(), 0.33, 2)  # book: ch09_body.tex "three indices near $0.33$"

# Independent scrambles: repeat the whole estimation R = 32 times, seeds 42..73.
reps_951 = np.array([jansen_S1(*jansen_with_values(qoi_951, M_951, lb_951, ub_951, seed=42 + r))
                     for r in range(R_951)])
assert np.array_equal(reps_951[0], S_hat_951)               # scramble 1 is the design printed above
assert M_951 == 256 and np.array_equal(reps_951[0], S_hat_951)  # book: ch09_body.tex "fresh design with $M = 256$; the design above is the first"
assert R_951 == 32 and len(reps_951) == 32  # book: ch09_body.tex "estimation under $R = 32$ independent scrambles"
assert np.allclose(np.round(reps_951[0], 2), [0.32, 0.33, 0.33])  # book: ch09_body.tex "1  & 0.32 & 0.33 & 0.33"
assert np.allclose(np.round(reps_951[1], 2), [0.33, 0.34, 0.33])  # book: ch09_body.tex "2  & 0.33 & 0.34 & 0.33"
assert np.allclose(np.round(reps_951[2], 2), [0.33, 0.33, 0.32])  # book: ch09_body.tex "3  & 0.33 & 0.33 & 0.32"
assert np.allclose(np.round(reps_951[3], 2), [0.34, 0.32, 0.34])  # book: ch09_body.tex "4  & 0.34 & 0.32 & 0.34"
assert R_951 == 32 and np.allclose(np.round(reps_951[R_951 - 1], 2), [0.35, 0.33, 0.34])  # book: ch09_body.tex "32 & 0.35 & 0.33 & 0.34"
mean_951 = reps_951.mean(axis=0)
s_951 = reps_951.std(axis=0, ddof=1)
t_951 = t_dist.ppf(T_LEVEL_951, R_951 - 1)
half_951 = t_951 * s_951 / np.sqrt(R_951)
ci_rep_951 = np.c_[mean_951 - half_951, mean_951 + half_951]
print('scramble replicates (R = 32, M = 256): mean ' + ', '.join(f'{v:.5f}' for v in mean_951)
      + ' | s ' + ', '.join(f'{v:.5f}' for v in s_951) + f' | t = {t_951:.4f}')
print('replicate 95% intervals: ' + '; '.join(f'[{a:.5f}, {b:.5f}]' for a, b in ci_rep_951))
assert R_951 - 1 == 31 and R_951 == 32 and agrees(T_LEVEL_951, 0.975, 3) and agrees(t_951, 2.04, 2)  # book: ch09_body.tex "where $t_{31,\,0.975} \approx 2.04$"
assert agrees(mean_951[0], 0.332, 3) and np.allclose(np.round(ci_rep_951[0], 3), [0.330, 0.335])  # book: ch09_body.tex "= 0.332 \quad [0.330,\ 0.335]"
assert agrees(mean_951[1], 0.332, 3) and np.allclose(np.round(ci_rep_951[1], 3), [0.329, 0.334])  # book: ch09_body.tex "= 0.332 \quad [0.329,\ 0.334]"
assert agrees(mean_951[2], 0.332, 3) and np.allclose(np.round(ci_rep_951[2], 3), [0.329, 0.335])  # book: ch09_body.tex "= 0.332 \quad [0.329,\ 0.335]"
assert np.all(ci_rep_951[:, 0] > 0)                                                # all three exclude zero
assert all(ci_rep_951[i, 0] < ci_rep_951[j, 1] for i in range(3) for j in range(3))  # and overlap
width_951 = ci_rep_951[:, 1] - ci_rep_951[:, 0]
print('replicate interval widths: ' + ', '.join(f'{w:.5f}' for w in width_951)
      + ' | width / index: ' + ', '.join(f'{w:.4f}' for w in width_951 / mean_951))
assert np.all(width_951 < 0.006) and np.all(100 * width_951 / mean_951 < 2)  # book: ch09_body.tex "each is under $0.006$ wide, less than $2\%$"
assert R_951 == 32 and M_951 == 256  # book: ch09_body.tex "That is what 32 scrambles of the $M = 256$ design buy"
assert R_951 * M_951 * (len(nominal_951) + 2) == 40960 and 32 * 1280 == 40960  # book: ch09_body.tex "at $32 \times 1280 = 40{,}960$ evaluations."

# The row bootstrap (nBoot = 100), the IID fallback, on scramble 1: rows of A and B resampled
# together; percentile intervals.  The scrambled rows are not IID, so these intervals are too wide.
boot_rng = np.random.default_rng(42)
boot = np.array([jansen_S1(fA_951[i], fB_951[i], fAB_951[:, i])
                 for i in (boot_rng.integers(0, M_951, M_951) for _ in range(NBOOT_951))])
ci_951 = np.percentile(boot, [2.5, 97.5], axis=0).T
assert np.allclose(np.round(ci_951[0], 2), [0.21, 0.42])  # book: ch09_body.tex "the fallback for IID rows, gives $[0.21,\ 0.42]$,"
assert (np.allclose(np.round(ci_951[1], 2), [0.20, 0.44]) and np.allclose(np.round(ci_951[2], 2), [0.25, 0.42])
        and NBOOT_951 == 100 and len(boot) == 100)  # book: ch09_body.tex "$[0.20,\ 0.44]$ and $[0.25,\ 0.42]$ on the first design (100 resamples)."
assert np.all(ci_951[:, 0] > 0)                                            # all three exclude zero
assert all(ci_951[i, 0] < ci_951[j, 1] for i in range(3) for j in range(3))  # and overlap
ratio_951 = (ci_951[:, 1] - ci_951[:, 0]) / (2 * t_951 * s_951)   # bootstrap width over the +/- t s one design gets
print('bootstrap widths over the replicate interval for one design: ' + ', '.join(f'{v:.3f}' for v in ratio_951))
assert np.all(ratio_951 > 1)                                      # the bootstrap is the wider, as the text says
assert (agrees(ratio_951.min(), 5, 0) and agrees(ratio_951.max(), 8, 0)
        and R_951 - 1 == 31 and agrees(T_LEVEL_951, 0.975, 3))  # book: ch09_body.tex "These are 5 to 8 times as wide as the interval $\pm t_{31,\,0.975}\,s$ that"

# Population first-order indices by Gauss-Legendre quadrature on the +/-50% box: equal.
x_gl, w_gl = np.polynomial.legendre.leggauss(200)
u_gl, w_gl = 1.0 + 0.5 * x_gl, w_gl / 2                    # U[0.5, 1.5] nodes and weights
g_c, g_b, g_t = (nominal_951[i] * u_gl for i in range(3))
Y_gl = peak_closed(g_c[:, None, None] * g_b[None, :, None] * g_t[None, None, :])
W_gl = w_gl[:, None, None] * w_gl[None, :, None] * w_gl[None, None, :]
EY_gl = np.sum(W_gl * Y_gl)
VY_gl = np.sum(W_gl * Y_gl ** 2) - EY_gl ** 2

def S1_population(axis):
    other = tuple(a for a in range(3) if a != axis)
    cond = np.sum(W_gl * Y_gl, axis=other) / w_gl            # E[Y | x_axis] at each node
    return (np.sum(w_gl * cond ** 2) - EY_gl ** 2) / VY_gl

S_pop_951 = np.array([S1_population(a) for a in range(3)])
print('population indices: ' + ', '.join(f'{s:.5f}' for s in S_pop_951))
SYM_TOL = 1e-12
assert np.ptp(S_pop_951) < SYM_TOL                          # R0 = c*beta*tau_R is symmetric

# §9.10 laboratory: share of the box with R0 = c*beta*tau_R <= 1, exactly, by quadrature.
from scipy.integrate import dblquad
_area = dblquad(lambda t, c: min(max((1 / (c * t) - lb[1]) / (ub[1] - lb[1]), 0), 1),
                lb[0], ub[0], lambda c: lb[2], lambda c: ub[2], epsabs=1e-12, epsrel=1e-12)[0]
share_R0_below_1 = _area / ((ub[0] - lb[0]) * (ub[2] - lb[2]))
assert agrees(100 * share_R0_below_1, 10.8, 1)  # book: ch09_body.tex "About $10.8\%$ of it"
print(f'PASS: part 5.  Laboratory box: {100 * share_R0_below_1:.2f}% has R0 <= 1.')

In [ ]:
# ── Verification Suite, part 6: Appendix F, the probability toolkit ─────────────────────
# Appendix F's numbers, recomputed where Chapter 9 uses them.  Three routes: exact rational
# arithmetic on the moments of U(0, 1), numerical quadrature, and a seeded Monte Carlo sample.
# Each check names the line of appendix_f.tex it is about with a comment of the form
#     # book: appendix_f.tex "text on that line"
# so the check accounts for that occurrence and for no other.  Parameters the appendix states
# are bound to named variables here, each with the value it prints.
from fractions import Fraction as Fr
from scipy.integrate import quad, dblquad

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

def moment_u01(k):
    """E[Q^k] for Q ~ U(0, 1), exactly: the integral of q^k over [0, 1]."""
    return Fr(1, k + 1)

QUAD_TOL = 1e-10                  # quadrature against the exact fractions
MC_TOL_REL = 2e-2                 # Monte Carlo against the exact values, relative
M_MC_F = 400_000
rng_f = np.random.default_rng(2026)

# Variance and standard deviation: Example F.1, then U(a, b)
m1, m2, m3 = moment_u01(1), moment_u01(2), moment_u01(3)
var_u = m2 - m1 ** 2
assert var_u * 12 == 1, 'FAIL: Var U(0, 1) = 1/12'
sd_u = np.sqrt(float(var_u))
assert agrees(sd_u, 0.289, 3)  # book: appendix_f.tex "\sigma = \frac{1}{\sqrt{12}} \approx 0.289"

BETA_NOM_F, HALF_WIDTH_F = 0.06, 0.5          # nominal beta of Section 9.5 and its +-50% range
lo_b, hi_b = BETA_NOM_F * (1 - HALF_WIDTH_F), BETA_NOM_F * (1 + HALF_WIDTH_F)
assert agrees(lo_b, 0.03, 2) and agrees(hi_b, 0.09, 2)  # book: appendix_f.tex "so it lies somewhere in $[0.03, 0.09]$"
mean_b, var_b = (lo_b + hi_b) / 2, (hi_b - lo_b) ** 2 / 12
sd_b = np.sqrt(var_b)
_mean_q = quad(lambda x: x / (hi_b - lo_b), lo_b, hi_b)[0]
_var_q = quad(lambda x: (x - mean_b) ** 2 / (hi_b - lo_b), lo_b, hi_b)[0]
assert abs(_mean_q - mean_b) < QUAD_TOL and abs(_var_q - var_b) < QUAD_TOL, 'FAIL: U(a, b) moments by quadrature'
_draws = rng_f.uniform(lo_b, hi_b, M_MC_F)
assert abs(_draws.var() / var_b - 1) < MC_TOL_REL, 'FAIL: U(a, b) variance by sampling'
assert agrees(lo_b, 0.03, 2) and agrees(hi_b, 0.09, 2) and agrees(mean_b, 0.06, 2)  # book: appendix_f.tex "$\beta \sim \mathcal{U}(0.03, 0.09)$ the mean is $0.06$"
assert agrees(hi_b - lo_b, 0.06, 2) and agrees(var_b, 0.0003, 4) and agrees(sd_b, 0.0173, 4)  # book: appendix_f.tex "is $(0.06)^2/12 = 0.0003$"
assert agrees(sd_b / mean_b, 0.289, 3) and agrees(sd_b / mean_b, sd_u, 12)  # book: appendix_f.tex "is about $0.289$. That is"

# Sums of independent inputs: the portfolio of Sections 8.4 and 9.1
S_PORT_F, SIGMA_PORT_F = (2, 1), (1, 3)       # weights s_i and standard deviations sigma_i
_terms = [(s * sg) ** 2 for s, sg in zip(S_PORT_F, SIGMA_PORT_F)]
var_port = sum(_terms)
assert _terms == [4, 9] and var_port == 13  # book: appendix_f.tex "variance of $4 + 9 = 13$"

# The half-square identity and the standard error of a mean of M = 100 draws
_q12 = rng_f.uniform(size=(M_MC_F, 2))
assert abs(np.mean((_q12[:, 0] - _q12[:, 1]) ** 2) * 6 - 1) < MC_TOL_REL, 'FAIL: E[(Y - Y\')^2] = 1/6'
M_SE_F = 100
se_100 = sd_u / np.sqrt(M_SE_F)
_means = rng_f.uniform(size=(40_000, M_SE_F)).mean(axis=1)
assert abs(_means.std() / se_100 - 1) < MC_TOL_REL, 'FAIL: standard error by sampling'
assert agrees(sd_u, 0.289, 3) and agrees(se_100, 0.029, 3)  # book: appendix_f.tex "$0.289/10 \approx 0.029$"
_shrink = (sd_u / np.sqrt(2 * M_SE_F)) / se_100
assert agrees(_shrink, 0.71, 2)  # book: appendix_f.tex "factor $1/\sqrt{2} \approx 0.71$"
assert np.isclose(sd_u / np.sqrt(4 * M_SE_F) * 2, se_100), 'FAIL: four times the samples halve the error'

# The delta method against the exact variance of R0 = c beta tau_R (the Chapter 8 self-check)
C_F, BETA_F, TAUR_F = 5, 0.06, 7               # c fixed; the means of beta and tau_R
SIGMA_BETA_F, SIGMA_TAUR_F = 0.02, 3           # their standard deviations, the two independent
var_delta = C_F ** 2 * (TAUR_F ** 2 * SIGMA_BETA_F ** 2 + BETA_F ** 2 * SIGMA_TAUR_F ** 2)
E_beta2, E_tau2 = BETA_F ** 2 + SIGMA_BETA_F ** 2, TAUR_F ** 2 + SIGMA_TAUR_F ** 2
var_exact = C_F ** 2 * (E_beta2 * E_tau2 - (BETA_F * TAUR_F) ** 2)
gap_F = C_F ** 2 * SIGMA_BETA_F ** 2 * SIGMA_TAUR_F ** 2
_b = rng_f.normal(BETA_F, SIGMA_BETA_F, M_MC_F)
_t = rng_f.normal(TAUR_F, SIGMA_TAUR_F, M_MC_F)
assert abs(np.var(C_F * _b * _t) / var_exact - 1) < MC_TOL_REL, 'FAIL: exact variance by sampling'
assert agrees(var_delta, 1.30, 2)  # book: appendix_f.tex "gives $\Var(\cR_0) \approx 1.30$"
assert C_F * C_F == 25 and agrees(E_beta2, 0.004, 3) and E_tau2 == 58 and agrees(BETA_F * TAUR_F, 0.42, 2) and agrees(var_exact, 1.39, 2)  # book: appendix_f.tex "= 25\,(0.004 \times 58 - 0.42^2) = 1.39$"
assert agrees(gap_F, 0.09, 2) and np.isclose(var_exact - var_delta, gap_F)  # book: appendix_f.tex "c^2\sigma_\beta^2\sigma_{\tau_R}^2 = 0.09$"

# Conditional expectation and the law of total variance: Y = Q1 + Q2 + Q1 Q2 (Section 9.4)
EY_F = m1 + m1 + m1 * m1
assert EY_F * 4 == 5  # book: appendix_f.tex "whose mean is $\E[Y] = 1/2 + 1/2 + 1/4 = 5/4$"

def cond_mean_F(q1):
    """E[Y | Q1 = q1]: Y averaged over Q2 alone, by quadrature."""
    return quad(lambda q2: q1 + q2 + q1 * q2, 0, 1)[0]

assert agrees(cond_mean_F(0), 0.5, 1) and agrees(cond_mean_F(1 / 2), 1.25, 2) and abs(cond_mean_F(1) - 2) < QUAD_TOL  # book: appendix_f.tex "which is $0.5$ at $q_1 = 0$, $1.25$ at"
slope_cm = 1 + m1                               # E[Y | Q1] = m1 + (1 + m1) Q1
D1_F = slope_cm ** 2 * var_u                    # Var[E(Y | Q1)]
E_1pq_sq = 1 + 2 * m1 + m2                      # E[(1 + Q1)^2]
EV1_F = E_1pq_sq * var_u                        # E[Var(Y | Q1)] = E[(1 + Q1)^2] Var(Q2)
var_Y_F = 2 * m2 + m2 * m2 + 2 * m1 * m1 + 4 * m1 * m2 - EY_F ** 2   # E[Y^2] - E[Y]^2, directly
_D1_q = quad(lambda q1: (cond_mean_F(q1) - float(EY_F)) ** 2, 0, 1)[0]
_EV1_q = quad(lambda q1: quad(lambda q2: (q1 + q2 + q1 * q2 - cond_mean_F(q1)) ** 2, 0, 1)[0], 0, 1)[0]
_VY_q = dblquad(lambda q2, q1: (q1 + q2 + q1 * q2 - float(EY_F)) ** 2, 0, 1, 0, 1)[0]
assert abs(_D1_q - float(D1_F)) < QUAD_TOL and abs(_EV1_q - float(EV1_F)) < QUAD_TOL and abs(_VY_q - float(var_Y_F)) < QUAD_TOL, 'FAIL: quadrature'
assert D1_F * 16 == 3 and D1_F * 144 == 27, 'FAIL: D1 = 3/16 = 27/144'
assert E_1pq_sq * 3 == 7  # book: appendix_f.tex "uses $\E[(1+Q_1)^2] = 1 + 1 + 1/3 = 7/3$, so"
assert EV1_F * 36 == 7 and EV1_F * 144 == 28  # book: appendix_f.tex "= (7/3)/12 = 7/36 = 28/144$"
assert var_Y_F * 144 == 55 and D1_F + EV1_F == var_Y_F  # book: appendix_f.tex "$27/144 + 28/144 = 55/144 = \Var(Y)$"
S1_F, ST2_F = D1_F / var_Y_F, EV1_F / var_Y_F
assert agrees(float(S1_F), 0.491, 3)  # book: appendix_f.tex "= \frac{27}{55} \approx 0.491"
assert agrees(float(ST2_F), 0.509, 3)  # book: appendix_f.tex "= \frac{28}{55} \approx 0.509"
EV2_F = E_1pq_sq * var_u                        # E[Var(Y | Q2)]: Y = q2 + Q1 (1 + q2) with Q2 fixed
ST1_F = EV2_F / var_Y_F
S12_F = ST1_F - S1_F
assert ST1_F * 55 == 28  # book: appendix_f.tex "By symmetry $\mathcal{S}_{T_1} = 28/55$ as well"
assert S1_F * 55 == 27 and S12_F * 55 == 1 and S12_F == var_u * var_u / var_Y_F  # book: appendix_f.tex "$\mathcal{S}_1 + \mathcal{S}_{12} = 27/55 + 1/55$"

# Covariance and correlation: the portfolio with rho = 0.5
RHO_PORT_F = 0.5
cov_port = RHO_PORT_F * SIGMA_PORT_F[0] * SIGMA_PORT_F[1]
var_corr = var_port + 2 * S_PORT_F[0] * S_PORT_F[1] * cov_port
_cv = [[SIGMA_PORT_F[0] ** 2, cov_port], [cov_port, SIGMA_PORT_F[1] ** 2]]
_qq = rng_f.multivariate_normal([0, 0], _cv, M_MC_F)
assert abs(np.var(S_PORT_F[0] * _qq[:, 0] + S_PORT_F[1] * _qq[:, 1]) / var_corr - 1) < MC_TOL_REL, 'FAIL: correlated portfolio by sampling'
assert agrees(cov_port, 1.5, 1)  # book: appendix_f.tex "$0.5 \times 1 \times 3 = 1.5$, and"
assert var_port == 13 and agrees(cov_port, 1.5, 1) and var_corr == 19  # book: appendix_f.tex "$\Var(Y) = 13 + 2 \times 2 \times 1 \times 1.5 = 19$"
assert agrees(np.sqrt(var_corr), 4.36, 2) and agrees(np.sqrt(var_port), 3.61, 2)  # book: appendix_f.tex "deviation is $\sqrt{19} \approx 4.36$ rather than $\sqrt{13} \approx 3.61$"
under_F = 1 - np.sqrt(var_port / var_corr)
assert round(100 * under_F) == 17  # book: appendix_f.tex "understates the output spread by about $17\%$"
cov_con = -var_u                                # Q2 = 1 - Q1: Cov = -Var(Q1), rho = -1
assert cov_con / var_u == -1 and var_u + var_u + 2 * cov_con == 0 and (var_u + var_u) * 6 == 1, 'FAIL: constraint correlation'
assert m3 - m2 + m1 * Fr(1, 4) - m1 * var_u == 0, 'FAIL: Cov(Q, (Q - 1/2)^2) = 0'

# A correlation is not a coupling coefficient: the least-squares slope on the relative scale
RHO_CB_F, CV_RATIO_F = 0.6, 0.5                 # rho between c and beta; beta half as variable as c
slope_rel = RHO_CB_F * CV_RATIO_F
CV_C_MC = 0.2                                   # any CV for c will do; the slope depends on the ratio
_z = rng_f.multivariate_normal([0, 0], [[1, RHO_CB_F], [RHO_CB_F, 1]], M_MC_F)
_slope_mc = np.polyfit(1 + CV_C_MC * _z[:, 0], 1 + CV_RATIO_F * CV_C_MC * _z[:, 1], 1)[0]
assert abs(_slope_mc / slope_rel - 1) < MC_TOL_REL, 'FAIL: slope by sampling'
assert agrees(slope_rel, 0.3, 1) and slope_rel < RHO_CB_F  # book: appendix_f.tex "the slope is $0.3$, not $0.6$"

# Self-check 1: tau_R ~ U(3.5, 10.5), +-50% about the nominal 7 days
TAU_RANGE_F = (3.5, 10.5)
a_t, b_t = TAU_RANGE_F
assert (a_t, b_t) == (TAUR_F * (1 - HALF_WIDTH_F), TAUR_F * (1 + HALF_WIDTH_F)), 'FAIL: +-50% about 7'
mean_t, width_t = (a_t + b_t) / 2, b_t - a_t
var_t = width_t ** 2 / 12
sd_t = np.sqrt(var_t)
assert mean_t == 7  # book: appendix_f.tex "\E[\tau_R] = (3.5 + 10.5)/2 = 7$~days and"
assert width_t == 7 and width_t * width_t == 49 and agrees(var_t, 4.08, 2)  # book: appendix_f.tex "$\Var(\tau_R) = 7^2/12 = 49/12 \approx 4.08$~days$^2$, so"
assert agrees(sd_t, 2.02, 2) and agrees(sd_t / mean_t, 0.289, 3) and agrees(round(sd_t, 2) / mean_t, 0.289, 3)  # book: appendix_f.tex "\sigma_{\tau_R} \approx 2.02$~days and $\CV = 2.02/7 \approx 0.289$."

# Self-check 2: Y = Q1 Q2
D1_p = m1 ** 2 * var_u                          # E[Y | Q1] = Q1 E[Q2] = Q1/2
EV1_p = m2 * var_u                              # Var(Y | Q1) = Q1^2 Var(Q2), averaged over Q1
var_p = m2 * m2 - (m1 * m1) ** 2                # E[Y^2] - E[Y]^2, directly
_VP_q = dblquad(lambda q2, q1: (q1 * q2 - float(m1 * m1)) ** 2, 0, 1, 0, 1)[0]
assert abs(_VP_q - float(var_p)) < QUAD_TOL, 'FAIL: Var(Q1 Q2) by quadrature'
assert D1_p * 48 == 1 and EV1_p * 36 == 1, 'FAIL: 1/48 and 1/36'
assert D1_p * 144 == 3 and EV1_p * 144 == 4 and (D1_p + EV1_p) * 144 == 7  # book: appendix_f.tex "(c) The two parts sum to $1/48 + 1/36 = 3/144 + 4/144 = 7/144$, and directly"
assert var_p * 144 == 7 and var_p == D1_p + EV1_p  # book: appendix_f.tex "$\Var(Y) = 1/9 - 1/16 = 7/144$ as well."
EV2_p = m2 * var_u                              # E[Var(Y | Q2)], by symmetry
S1_p, ST1_p = D1_p / var_p, EV2_p / var_p
S12_p = ST1_p - S1_p
assert S1_p * 7 == 3 and agrees(float(S1_p), 0.429, 3)  # book: appendix_f.tex "= (3/144)/(7/144) = 3/7 \approx 0.429$, and"
assert EV2_p * 144 == 4 and ST1_p * 7 == 4 and agrees(float(ST1_p), 0.571, 3)  # book: appendix_f.tex "= 4/144$, so $\mathcal{S}_{T_1} = 4/7 \approx 0.571$."
assert S12_p * 7 == 1 and agrees(float(S12_p), 0.143, 3)  # book: appendix_f.tex "\mathcal{S}_{T_1} - \mathcal{S}_1 = 1/7 \approx 0.143$, and"
assert (S1_p * 7, S1_p * 7, S12_p * 7) == (3, 3, 1) and S1_p + S1_p + S12_p == 1  # book: appendix_f.tex "$\mathcal{S}_1 + \mathcal{S}_2 + \mathcal{S}_{12} = 3/7 + 3/7 + 1/7 = 1$."

# Self-check 3: the portfolio with rho = -0.5
RHO_NEG_F = -0.5
cov_neg = RHO_NEG_F * SIGMA_PORT_F[0] * SIGMA_PORT_F[1]
var_neg = _terms[0] + _terms[1] + 2 * S_PORT_F[0] * S_PORT_F[1] * cov_neg
assert agrees(cov_neg, -1.5, 1)  # book: appendix_f.tex "The covariance is $-0.5 \times 1 \times 3 = -1.5$, so"
assert _terms[0] == 4 and agrees(cov_neg, -1.5, 1) and var_neg == 7  # book: appendix_f.tex "\Var(Y) = 4 + 9 + 2 \times 2 \times 1 \times (-1.5) = 7$ and"
assert agrees(np.sqrt(var_neg), 2.65, 2) and agrees(np.sqrt(var_port), 3.61, 2) and var_neg < var_port  # book: appendix_f.tex "$\sigma_Y = \sqrt{7} \approx 2.65$, against $\sqrt{13} \approx 3.61$ for"
print(f'PASS: part 6, Appendix F.  S_1 = {S1_F} = {float(S1_F):.3f}; Var(R0): delta {var_delta:.2f}, '
      f'exact {var_exact:.2f}; sigma_Y {np.sqrt(var_port):.2f} -> {np.sqrt(var_corr):.2f} (rho = 0.5)')

In [ ]:
# ── Verification Suite, part 7: gradient-based global measures ─────────────────
# The section "Gradient-Based Global Measures": DGSMs, the bound on the total index, and
# the active subspace of the SIR example of the confidence-interval worked example
# (c, beta, tau_R uniform on +/-50% about 8, 0.06 and 10 days; S(0) = 0.999, I(0) = 0.001).
# Sources of the definitions: smith2024uncertainty eqs. (9.39)-(9.40) and (10.10),
# Algorithm 10.14; sullivan2015introduction Section 10.5 and Exercise 10.9.
# In order, the cell establishes:
#   1. nu_i by quadrature, the bound (b - a)^2 nu_i / (pi^2 D), and the total indices;
#   2. the Monte Carlo nu_i from the 256 rows of A (seed 42) against quadrature;
#   3. the active subspace of I_max from black-box gradients (ODE + central differences
#      in log coordinates): rank one, w1 = (1,1,1)/sqrt(3);
#   4. the active subspace of the time of the peak: rank two, null vector (1,-1,0)/sqrt(2),
#      and the rank, not the eigenvalues, stable across scrambles.
def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

def agrees_sig(x, shown, nsig):
    """True when x and the printed value agree to nsig significant digits."""
    return abs(x - shown) <= 0.5 * 10.0 ** (np.floor(np.log10(abs(shown))) - nsig + 1) + 1e-15

nominal_gg = np.array([8.0, 0.06, 10.0])
lb_gg, ub_gg = 0.5 * nominal_gg, 1.5 * nominal_gg
S0_gg, I0_gg = 0.999, 0.001
M_gg, H_gg = 256, 1e-4
assert I0_gg == 0.001 and S0_gg + I0_gg == 1  # book: ch09_body.tex "the epidemic starts from $I(0) = 0.001$"
assert np.array_equal(nominal_gg, nominal_951) and nominal_gg[0] == 8 and agrees(nominal_gg[1], 0.06, 2) and 100 * ub_gg[0] / nominal_gg[0] - 100 == 50  # book: ch09_body.tex "uniform over $\pm 50\%$ about $8$, $0.06$ and"
assert nominal_gg[2] == 10   # stated inputs printed in the text

def peak_gg(R0):
    R0 = np.asarray(R0, float)
    return np.where(R0 * S0_gg > 1, I0_gg + S0_gg - (1 + np.log(R0 * S0_gg)) / R0, I0_gg)

def dpeak_gg(R0):
    """dI_max/dR0 of the closed form: ln(R0 S0)/R0^2 above threshold, 0 below."""
    R0 = np.asarray(R0, float)
    return np.where(R0 * S0_gg > 1, np.log(R0 * S0_gg) / R0 ** 2, 0.0)

# 1. Quadrature: Gauss-Legendre, 200 nodes per axis, on U[0.5, 1.5] times the nominal values.
xq, wq = np.polynomial.legendre.leggauss(200)
uq, wq = 1.0 + 0.5 * xq, wq / 2
gq = [nominal_gg[i] * uq for i in range(3)]
Rq = gq[0][:, None, None] * gq[1][None, :, None] * gq[2][None, None, :]
Wq = wq[:, None, None] * wq[None, :, None] * wq[None, None, :]
Yq = peak_gg(Rq)
EYq = np.sum(Wq * Yq)
Dq = np.sum(Wq * Yq ** 2) - EYq ** 2
shape = lambda i: [-1 if a == i else 1 for a in range(3)]
nu_q = np.array([np.sum(Wq * (dpeak_gg(Rq) * Rq / gq[i].reshape(shape(i))) ** 2) for i in range(3)])
width_gg = ub_gg - lb_gg
bound_q = width_gg ** 2 * nu_q / (np.pi ** 2 * Dq)
ST_q = []
for i in range(3):
    wi = wq.reshape(shape(i))
    cond_var = np.sum(wi * (Yq - np.sum(wi * Yq, axis=i, keepdims=True)) ** 2, axis=i)
    wo = [wq for a in range(3) if a != i]
    ST_q.append(np.sum(wo[0][:, None] * wo[1][None, :] * cond_var) / Dq)
ST_q = np.array(ST_q)
print('nu (quadrature): ' + ', '.join(f'{v:.5g}' for v in nu_q))
print('DGSM bounds: ' + ', '.join(f'{v:.5f}' for v in bound_q) + ' | total indices: '
      + ', '.join(f'{v:.5f}' for v in ST_q))
assert agrees_sig(nu_q[0], 2.21e-3, 3) and agrees(nu_q[1], 39.2, 1) and agrees_sig(nu_q[2], 1.41e-3, 3)  # book: ch09_body.tex "$\vartheta_c = 2.21\times10^{-3}$, $\vartheta_\beta = 39.2$ and $\vartheta_{\tau_R} = 1.41\times10^{-3}$."
assert 3 < np.log10(nu_q[1] / nu_q.min()) < 5                 # four orders of magnitude apart
assert np.ptp(bound_q) < 1e-12 and agrees(bound_q[0], 0.485, 3)  # book: ch09_body.tex "and the three bounds come out equal, at $0.485$."
assert np.allclose(width_gg, nominal_gg)                      # each range is as wide as its nominal value
assert np.ptp(ST_q) < 1e-12 and agrees(ST_q[0], 0.336, 3)  # book: ch09_body.tex "same box, are $\mathcal{S}_{T_i} = 0.336$ for all three."
assert np.all(ST_q <= bound_q) and agrees(bound_q[0] / ST_q[0], 1.4, 1)  # book: ch09_body.tex "factor of about $1.4$ to spare."
assert np.allclose(ST_q, S_pop_951, atol=0.01) and np.all(ST_q >= S_pop_951)   # S_T >= S_i, close: weak interaction

# 2. Black-box gradients by ODE (event at dI/dt = 0) and central differences in x = log p.
def peak_time_gg(p):
    c, b, tau = p
    if c * b * tau * S0_gg <= 1:
        return I0_gg, 0.0
    ev = lambda t, y: c * b * y[0] - 1.0 / tau
    ev.terminal, ev.direction = True, -1
    s = solve_ivp(lambda t, y: [-c * b * y[0] * y[1], c * b * y[0] * y[1] - y[1] / tau],
                  [0, 5000], [S0_gg, I0_gg], events=ev, method='DOP853', rtol=1e-12, atol=1e-14)
    return s.y_events[0][0][1], s.t_events[0][0]

def grad_log_gg(p, h=H_gg):
    G, x = np.zeros((2, 3)), np.log(p)
    for i in range(3):
        xp, xm = x.copy(), x.copy()
        xp[i] += h
        xm[i] -= h
        G[:, i] = (np.array(peak_time_gg(np.exp(xp))) - np.array(peak_time_gg(np.exp(xm)))) / (2 * h)
    return G

def active_gg(G):
    lam, W = np.linalg.eigh(G.T @ G / len(G))
    lam, W = lam[::-1], W[:, ::-1]
    W[:, 0] *= np.sign(W[:, 0].sum())
    return lam, W

A_gg, _ = saltelli_sample(M_gg, lb_gg, ub_gg, 42)            # the matrix A of the first design
assert M_gg == 256 and len(A_gg) == 256   # stated inputs printed in the text
G_gg = np.array([grad_log_gg(a) for a in A_gg])               # (256, 2, 3): I_max and t_peak
R0_gg = A_gg.prod(axis=1)
assert np.abs(G_gg[:, 0, :] - (dpeak_gg(R0_gg) * R0_gg)[:, None]).max() < 1e-7   # ODE gradient = closed form
nu_mc = np.mean((G_gg[:, 0, :] / A_gg) ** 2, axis=0)          # d/dp = (d/dlog p) / p
print('nu (256 rows of A): ' + ', '.join(f'{v:.5g}' for v in nu_mc)
      + ' | relative to quadrature: ' + ', '.join(f'{v:.4f}' for v in nu_mc / nu_q))
assert np.all(np.abs(nu_mc / nu_q - 1) < 1 / 100)  # book: ch09_body.tex "$A$ alone agree with the quadrature values to within $1\%$."

# 3. I_max: rank one, w1 = (1,1,1)/sqrt(3).
lamI_gg, WI_gg = active_gg(G_gg[:, 0, :])
print('I_max: lambda = ' + ', '.join(f'{v:.4g}' for v in lamI_gg) + ' | w1 = ' + str(WI_gg[:, 0]))
assert M_gg == 256   # stated inputs printed in the text
assert agrees(lamI_gg[0], 0.306, 3) and np.all(np.abs(lamI_gg[1:]) < 1e-15 * lamI_gg[0])  # book: ch09_body.tex "$\lambda_1 = 0.306$, with $\lambda_2$ and $\lambda_3$ below $10^{-15}\lambda_1$,"
assert np.abs(WI_gg[:, 0] - 1 / np.sqrt(3)).max() < 0.5e-12   # (1,1,1)/sqrt(3) to twelve decimal places
assert agrees(3 * np.sum(Wq * (dpeak_gg(Rq) * Rq) ** 2), lamI_gg[0], 2)   # population lambda_1 = 3 E[(R0 f')^2]

# 4. Time of the peak: rank two, null vector (1,-1,0)/sqrt(2).
lamT_gg, WT_gg = active_gg(G_gg[:, 1, :])
print('t_peak: lambda = ' + ', '.join(f'{v:.4g}' for v in lamT_gg) + f' | lambda2/lambda1 = {lamT_gg[1] / lamT_gg[0]:.4f}'
      + ' | w1 = ' + str(WT_gg[:, 0]) + ' | w3 = ' + str(WT_gg[:, 2]))
assert M_gg == 256 and agrees(lamT_gg[1] / lamT_gg[0], 0.013, 3)  # book: ch09_body.tex "$\lambda_2/\lambda_1 = 0.013$, small but far above round-off, while"
assert abs(lamT_gg[2]) < 1e-15 * lamT_gg[0]  # book: ch09_body.tex "$\lambda_3$ is again below $10^{-15}\lambda_1$, and $\vec{w}_3$ is"
assert np.allclose(np.abs(WT_gg[:, 2]), [2 ** -0.5, 2 ** -0.5, 0], atol=1e-10) and WT_gg[0, 2] * WT_gg[1, 2] < 0
assert np.allclose(np.round(WT_gg[:, 0], 2), [0.68, 0.68, 0.28])  # book: ch09_body.tex "$\vec{w}_1 = (0.68, 0.68, 0.28)$ leans toward $\tau_R$."
# the gradient lies in span{(1,1,1), (0,0,1)}: residual after projecting on that span is round-off
P_span = np.linalg.qr(np.array([[1.0, 1, 1], [0, 0, 1]]).T)[0]
resid = G_gg[:, 1, :] - (G_gg[:, 1, :] @ P_span) @ P_span.T
assert np.abs(resid).max() < 1e-6 * np.abs(G_gg[:, 1, :]).max()
# Other scrambles: lambda_1 moves (rows near threshold dominate), the rank does not.
lam_seeds = [active_gg(np.array([grad_log_gg(a)[1] for a in saltelli_sample(M_gg, lb_gg, ub_gg, s)[0]]))[0]
             for s in (43, 44)]
lam1_all = np.array([lamT_gg[0]] + [l[0] for l in lam_seeds])
print('t_peak lambda_1, seeds 42-44: ' + ', '.join(f'{v:.4g}' for v in lam1_all))
assert lam1_all.max() / lam1_all.min() > 2                    # eigenvalues change from scramble to scramble
assert all(abs(l[2]) < 1e-15 * l[0] and l[1] > 1e-3 * l[0] for l in lam_seeds)   # rank two each time
# Figure fig_activesub: the generator draws from this same design and these same gradients.
print('PASS: part 7 (gradient-based global measures).')

In [ ]:
# ── Verification Suite, part 8: check that the fixed factor tau_m does not matter ──
# The Morris workflow fixes tau_m after screening.  Following smith2024uncertainty Remark 8.23
# and Section 9.6.1, sample I_max with tau_m random and with tau_m fixed at its nominal value,
# and compare the two distributions (two-sample Kolmogorov-Smirnov test, scipy.stats.ks_2samp).
# Model and QOI exactly as the fig_morris generator: SIR with slow demography, N = 1000, one
# initial case, peak count over 90 days on a 4000-point grid of the dense output; nominal
# (5, 0.06, 7, 10000), ranges +/-50%.  Two independent samples of 1000 runs from one generator,
# default_rng(42): first all four POIs, then (c, beta, tau_R) with tau_m = 10000.  The figure
# generator figures/generators/generate_fig_fixedfactor.py draws the same samples.
from scipy.stats import ks_2samp

def agrees(x, shown, ndigits):
    """True when x rounds to the value the book prints, at the book's precision."""
    return abs(x - shown) <= 0.5 * 10.0 ** (-ndigits) + 1e-12

N_ff, T_ff = 1000.0, 90.0
nominal_ff = np.array([5.0, 0.06, 7.0, 10000.0])
lb_ff, ub_ff = 0.5 * nominal_ff, 1.5 * nominal_ff
M_ff, SEED_ff = 1000, 42

def peak_count_ff(c, beta, tau_R, tau_m):
    """Peak infectious count over a 90-day SIR run with slow demography (as fig_morris)."""
    gamma_R, nu_m = 1.0 / tau_R, 1.0 / tau_m
    def rhs(t, y):
        S, I, R = y
        lam = c * beta * I / N_ff
        return [nu_m * N_ff - lam * S - nu_m * S, lam * S - gamma_R * I - nu_m * I,
                gamma_R * I - nu_m * R]
    sol = solve_ivp(rhs, [0, T_ff], [N_ff - 1.0, 1.0, 0.0], dense_output=True,
                    rtol=1e-9, atol=1e-11, max_step=0.5)
    return sol.sol(np.linspace(0, T_ff, 4000))[1].max()

rng_ff = np.random.default_rng(SEED_ff)
X_full_ff = rng_ff.uniform(lb_ff, ub_ff, (M_ff, 4))
X_fix_ff = rng_ff.uniform(lb_ff[:3], ub_ff[:3], (M_ff, 3))
Y_full_ff = np.array([peak_count_ff(*x) for x in X_full_ff])
Y_fix_ff = np.array([peak_count_ff(*x, nominal_ff[3]) for x in X_fix_ff])
Y_pair_ff = np.array([peak_count_ff(*x[:3], nominal_ff[3]) for x in X_full_ff])
ks_ff = ks_2samp(Y_full_ff, Y_fix_ff)
dmax_ff = np.abs(Y_full_ff - Y_pair_ff).max()
print(f'KS statistic = {ks_ff.statistic:.4f}, p = {ks_ff.pvalue:.4f}; largest paired change = {dmax_ff:.4f} persons')
print(f'runs with no outbreak (I_max = 1): {np.sum(Y_full_ff == 1.0)} and {np.sum(Y_fix_ff == 1.0)} of {M_ff}')

assert M_ff == 1000 and 100 * ub_ff[0] / nominal_ff[0] - 100 == 50   # stated inputs printed in the text
assert nominal_ff[3] == 10000 and SEED_ff == 42   # stated inputs printed in the text
assert M_ff == 1000 and len(Y_full_ff) == 1000  # book: ch09_body.tex "In the setting of Figure~\ref{fig:morris}, draw $M = 1000$ runs with all four"
assert agrees(ks_ff.statistic, 0.044, 3) and agrees(ks_ff.pvalue, 0.29, 2)  # book: ch09_body.tex "empirical distribution functions, is $0.044$, with $p = 0.29$. The cited"
assert ks_ff.pvalue > 0.05                                    # no difference at the 5% level
assert dmax_ff < 0.2  # book: ch09_body.tex "$0.2$ persons. Fixing $\tau_m$ at its nominal value changes the distribution of"
assert T_ff == 90   # stated inputs printed in the text
assert T_ff == 90 and M_ff == 1000   # stated inputs printed in the text
assert nominal_ff[3] == 10000   # stated inputs printed in the text
assert agrees(ks_ff.pvalue, 0.29, 2)  # book: ch09_body.tex "The two-sample Kolmogorov--Smirnov test detects no difference ($p = 0.29$).}"
# The tall first bin [0, 20) of the figure: small outbreaks and the runs with no outbreak.
assert np.mean(Y_full_ff < 20) > 2 * np.mean((Y_full_ff >= 20) & (Y_full_ff < 40))
print('PASS: part 8 (fixing tau_m leaves the distribution of I_max unchanged).')